In [2]:
import pandas as pd
df = pd.read_csv("../data/raw/estat_env_air_gge_en.csv.gz", compression="gzip")

df.info()

/var/folders/vy/cp516jcd6sx956mt5l6mqtjr0000gn/T/ipykernel_30325/357650269.py:2: DtypeWarning: Columns (0: CONF_STATUS) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/estat_env_air_gge_en.csv.gz", compression="gzip")


<class 'pandas.DataFrame'>
RangeIndex: 1592910 entries, 0 to 1592909
Data columns (total 11 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   DATAFLOW     1592910 non-null  str    
 1   LAST UPDATE  1592910 non-null  str    
 2   freq         1592910 non-null  str    
 3   unit         1592910 non-null  str    
 4   airpol       1592910 non-null  str    
 5   src_crf      1592910 non-null  str    
 6   geo          1592910 non-null  str    
 7   TIME_PERIOD  1592910 non-null  int64  
 8   OBS_VALUE    1520160 non-null  float64
 9   OBS_FLAG     120964 non-null   str    
 10  CONF_STATUS  460 non-null      str    
dtypes: float64(1), int64(1), str(9)
memory usage: 133.7 MB


same - trash

In [4]:
df["freq"].value_counts()

freq
A    1592910
Name: count, dtype: int64

same - trash

In [6]:
df["DATAFLOW"].value_counts()

DATAFLOW
ESTAT:ENV_AIR_GGE(1.0)    1592910
Name: count, dtype: int64

same - trash

In [7]:
df["LAST UPDATE"].value_counts()

LAST UPDATE
08/05/25 23:00:00    1592910
Name: count, dtype: int64

In [ ]:
df["unit"].value_counts() ##странновао их ровно по ровну

unit
MIO_T    796455
THS_T    796455
Name: count, dtype: int64

__________________________________________________________________________________________________________________

получается что в выборке имеем по два раза записанные одианковые данные в разных еденицах измерения 

In [11]:
cols_without_unit = [
    'DATAFLOW', 'LAST UPDATE', 'freq', 'airpol',
    'src_crf', 'geo', 'TIME_PERIOD']
res = (df.groupby(cols_without_unit)["unit"].nunique().value_counts())
print(res)

unit
2    796455
Name: count, dtype: int64


__________________________________________________________________________________________________________________
подвтерждаем гепотизу сравниваем значения выясняем что на 796455 индексе начианется измерение в тысячах

In [31]:
print(df.index[df["OBS_VALUE"] == 0.05434])
print(df.index[df["OBS_VALUE"] == 0.05434*1000])
print(df.index[df["unit"] == "THS_T"])
print(df["OBS_VALUE"][df.index == 796455])
a = df.loc[1]["OBS_VALUE"]
b = df.loc[796456]["OBS_VALUE"]
print(b/a)

Index([0, 88937, 89957, 187355, 293910, 401804], dtype='int64')
RangeIndex(start=0, stop=0, step=1)
RangeIndex(start=796455, stop=1592910, step=1)
796455    54.34
Name: OBS_VALUE, dtype: float64
1000.0000000000001


__________________________________________________________________________________________________________________
фиксируем пропуски, так же подтерждаем что их по ровну следоватлеьно ячейки симметрично равны \n upd - нужно будет либо удалить половину либо разделить и замрежить с данными того и того типа предварительно убрав флаг распреддения 

In [38]:
print(df["OBS_VALUE"][df["unit"] == "THS_T"].isna().sum())
df["OBS_VALUE"][df["unit"] == "MIO_T"].isna().sum()


36375


np.int64(36375)

__________________________________________________________________________________________________________________
пропуски отсутсвуют данные важные вероятнее всего точно так же продублированы в разных эквивалентах

In [42]:
df["airpol"].value_counts()
df["airpol"].isna().sum()

np.int64(0)

__________________________________________________________________________________________________________________
важные даннве маркировка источника загрязнений - требует разбирательства и ознкаомления с принципами маркерковк

In [43]:
df["src_crf"].value_counts()

src_crf
CRF2      22438
CRF2B     21272
CRF2G     21210
CRF2C     20808
CRF2H     19440
          ...  
CRF2B6     2286
CRF5F1     2108
CRF1C      2096
CRF5F2     2040
CRF5F3     1836
Name: count, Length: 166, dtype: int64

__________________________________________________________________________________________________________________
полезные данные о странах происхождения загрезнений 

In [47]:
df["geo"].value_counts()

geo
EU27_2020    59324
RO           56496
DE           56064
ES           55990
LT           55872
IE           54902
NO           54544
SI           54402
PT           54086
LV           54026
BE           53880
BG           53692
PL           53610
HU           53570
HR           53478
LU           53210
CZ           53172
FI           53046
IT           53022
NL           52876
FR           52352
EE           51760
IS           51666
CY           51632
SK           51600
AT           49626
MT           49466
DK           49378
EL           49320
SE           46848
Name: count, dtype: int64

__________________________________________________________________________________________________________________
период времени с 1990 года по 2023 год(хотя заявлено что полсденее обновление было в 2025)
препдоложительно все выбросы считаются за год

In [49]:
df["TIME_PERIOD"].value_counts().sort_index()

TIME_PERIOD
1990    46516
1991    46534
1992    46600
1993    46636
1994    46680
1995    46782
1996    46810
1997    46828
1998    46848
1999    46860
2000    46880
2001    46900
2002    46900
2003    46912
2004    46918
2005    46930
2006    46948
2007    46928
2008    46940
2009    46930
2010    46934
2011    46932
2012    46940
2013    46936
2014    46940
2015    46920
2016    46924
2017    46924
2018    46924
2019    46916
2020    46912
2021    46902
2022    46896
2023    46630
Name: count, dtype: int64

In [51]:
df["OBS_FLAG"].value_counts().sort_index()

OBS_FLAG
d    56310
m    64654
Name: count, dtype: int64

__________________________________________________________________________________________________________________
если с m  все понятно и почти гарантированно оно нам не нужно то с d следует разбираться в дальнейшем актвинее

In [55]:
df[df["OBS_FLAG"] == "m"][["airpol", "src_crf", "geo", "TIME_PERIOD", "OBS_VALUE"]].head(20)
print(df[df["OBS_FLAG"] == "m"]["OBS_VALUE"].isna().value_counts())
print(df[df["OBS_FLAG"] == "d"]["OBS_VALUE"].isna().value_counts())

OBS_VALUE
True    64654
Name: count, dtype: int64
OBS_VALUE
False    48674
True      7636
Name: count, dtype: int64


__________________________________________________________________________________________________________________
засекреченные данные можно удалять

In [58]:
df["CONF_STATUS"].value_counts()
df[df["CONF_STATUS"] == "C"]["OBS_VALUE"].isna().value_counts()

OBS_VALUE
True    460
Name: count, dtype: int64